In [0]:
from pyspark.sql import functions as F

silver = spark.table("aws_s3.ventas.silver_ventas")
rechazadas = spark.table("aws_s3.ventas.silver_ventas_rechazadas")
gold_diarias = spark.table("aws_s3.ventas.gold_ventas_diarias")

print("Tablas cargadas correctamente.")

In [0]:
# 1. Integridad de los campos obligatorios
nulos_obligatorios = silver.filter(
    F.col("id_venta").isNull() |
    F.col("id_cliente").isNull() |
    F.col("fecha_normalizada").isNull()
).count()

# 2. Identificadores de venta duplicados
duplicados = (
    silver.groupBy("id_venta")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Cantidades no válidas
cantidades_invalidas = silver.filter(
    F.col("cantidad_limpia").isNull() |
    (F.col("cantidad_limpia") <= 0)
).count()

# 4. Descuentos fuera del rango permitido
descuentos_invalidos = silver.filter(
    F.col("descuento_limpio").isNull() |
    (F.col("descuento_limpio") < 0) |
    (F.col("descuento_limpio") > 100)
).count()

# 5. Precios unitarios inválidos
precios_invalidos = silver.filter(
    F.col("precio_unitario_crc_limpio").isNull() |
    (F.col("precio_unitario_crc_limpio") < 0)
).count()

# 6. Registros con total original que requiere revisión
totales_por_revisar = silver.filter(
    F.col("total_revisar") == True
).count()

# 7. Ingresos calculados desde Silver
ingresos_silver = silver.agg(
    F.round(F.sum("total_esperado"), 2).alias("total")
).first()["total"] or 0.0

# 8. Ingresos agregados en Gold
ingresos_gold = gold_diarias.agg(
    F.round(F.sum("ingresos_crc"), 2).alias("total")
).first()["total"] or 0.0

diferencia_ingresos = abs(float(ingresos_silver) - float(ingresos_gold))

# 9. Cantidad de registros rechazados
cantidad_rechazados = rechazadas.count()

# Construir resultados
resultados = [
    ("Campos obligatorios", nulos_obligatorios == 0,
     nulos_obligatorios, "Esperado: 0 registros con nulos"),

    ("Ventas duplicadas", duplicados == 0,
     duplicados, "Esperado: 0 identificadores duplicados"),

    ("Cantidades inválidas", cantidades_invalidas == 0,
     cantidades_invalidas, "Esperado: 0 cantidades inválidas"),

    ("Descuentos inválidos", descuentos_invalidos == 0,
     descuentos_invalidos, "Esperado: 0 descuentos fuera de rango"),

    ("Precios inválidos", precios_invalidos == 0,
     precios_invalidos, "Esperado: 0 precios inválidos"),

    ("Conciliación Silver-Gold", diferencia_ingresos < 0.02,
     round(diferencia_ingresos, 2), "Esperado: diferencia menor a ₡0.02"),
]

# Crear un DataFrame de resultados
df_resultados = spark.createDataFrame(
    [
        (
            nombre,
            "PASS" if correcto else "FAIL",
            str(valor),
            detalle
        )
        for nombre, correcto, valor, detalle in resultados
    ],
    ["prueba", "estado", "valor_obtenido", "criterio"]
)

display(df_resultados)

In [0]:
# Revisar los registros que requieren atención
print("Registros rechazados:", cantidad_rechazados)
print("Totales por revisar:", totales_por_revisar)

display(
    silver
    .filter(F.col("total_revisar") == True)
    .select(
        "id_venta",
        "total_crc_limpio",
        "total_esperado",
        "total_revisar"
    )
    .limit(20)
)

In [0]:
# Guardar los resultados de calidad
df_resultados.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aws_s3.ventas.resultados_calidad")

display(spark.table("aws_s3.ventas.resultados_calidad"))